# 自助法完整教程：重抽样的力量

## 📚 教学目标
1. 理解自助法的基本原理（重抽样）
2. 掌握自助法置信区间的构建步骤
3. 用 Python 实现自助法
4. 对比自助法与传统方法的结果
5. 理解自助法的优势和适用场景

**参考书**：《基础统计学(第14版)》（Triola）第 7-4 节
**教学策略**：先用微型数据集演示重抽样的每一步，再扩展到真实场景，最后对比传统方法

---

## 1. 场景设定：为什么需要自助法？

### 🎯 传统方法的局限

前面学的置信区间方法都有严格的假设：
- 比例估计：需要 $np \geq 5$ 且 $nq \geq 5$
- 均值估计：需要总体正态或 $n > 30$
- 方差估计：**必须**总体正态

**如果这些条件不满足怎么办？**

### 💡 自助法的核心思想

自助法（Bootstrap）的核心思想非常简单：

> **用样本数据模拟总体分布，通过反复重抽样来估计统计量的变异。**

就像从一个袋子里反复摸球，每次摸完放回去（有放回抽样），通过大量重复来了解袋子里面的情况。

### 📖 书中的定义

> 通过统计软件对样本数据进行多次「重采样」（如 1000 次），然后通过这 1000 个排好序的结果求得置信区间。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print('✅ 库导入完成')

---

## 2. 自助法的基本原理

### 📐 自助法的步骤

1. **原始样本**：有 $n$ 个数据点
2. **重抽样**：从原始样本中**有放回地**随机抽取 $n$ 个数据点（形成一个自助样本）
3. **计算统计量**：计算自助样本的统计量（均值、中位数、方差等）
4. **重复**：重复步骤 2-3 大量次数（如 1000 次或 10000 次）
5. **构建置信区间**：用这 1000 个统计量的分布来构建置信区间

### 💡 为什么「有放回」很重要？

有放回抽样意味着同一个数据点可能在同一个自助样本中出现多次，也可能一次都不出现。
这创造了数据的「变异」，模拟了从总体中抽样的过程。

---

## 3. 微型数据演示：看见每一步

### 🎯 场景

假设我们只有 10 个学生的学习时间数据（小时/周），想要估计平均学习时间的置信区间。

In [ ]:
# ========== 微型数据集 ==========
study_hours = np.array([3, 5, 7, 2, 8, 4, 6, 5, 3, 7])
n_micro = len(study_hours)

print('📋 微型数据集：学生学习时间（小时/周）')
print(f'  数据: {study_hours}')
print(f'  样本量 n = {n_micro}')
print(f'  样本均值 x̄ = {np.mean(study_hours):.2f}')
print(f'  样本标准差 s = {np.std(study_hours, ddof=1):.2f}')

In [ ]:
# ========== 步骤 1: 一次重抽样演示 ==========
print('📊 步骤 1: 一次重抽样的过程')
print(f'\n  原始数据: {study_hours}')

# 有放回地抽取 n 个数据点
bootstrap_sample_1 = np.random.choice(study_hours, size=n_micro, replace=True)
print(f'  自助样本1: {bootstrap_sample_1}')
print(f'  均值 = {np.mean(bootstrap_sample_1):.2f}')

bootstrap_sample_2 = np.random.choice(study_hours, size=n_micro, replace=True)
print(f'\n  自助样本2: {bootstrap_sample_2}')
print(f'  均值 = {np.mean(bootstrap_sample_2):.2f}')

bootstrap_sample_3 = np.random.choice(study_hours, size=n_micro, replace=True)
print(f'\n  自助样本3: {bootstrap_sample_3}')
print(f'  均值 = {np.mean(bootstrap_sample_3):.2f}')

print(f'\n💡 注意：')
print(f'  - 同一个数字可能出现多次（如某个学生被多次选中）')
print(f'  - 某些数字可能不出现（如某个学生未被选中）')
print(f'  - 每次抽样的均值都不同，这就是自助法的「变异」')

In [ ]:
# ========== 步骤 2: 大量重抽样 ==========
n_bootstrap = 10000

# 计算每个自助样本的均值
bootstrap_means = np.array([
    np.mean(np.random.choice(study_hours, size=n_micro, replace=True))
    for _ in range(n_bootstrap)
])

print(f'📊 步骤 2: {n_bootstrap} 次重抽样')
print(f'  自助样本均值的均值 = {np.mean(bootstrap_means):.4f}')
print(f'  自助样本均值的标准差 = {np.std(bootstrap_means):.4f}')
print(f'  原始样本均值 = {np.mean(study_hours):.4f}')
print(f'  💡 自助样本均值的均值 ≈ 原始样本均值')

In [ ]:
# ========== 步骤 3: 构建置信区间（百分位数法） ==========
alpha = 0.05
ci_lower = np.percentile(bootstrap_means, 100 * alpha / 2)
ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))

print(f'📊 步骤 3: 构建 95% 置信区间（百分位数法）')
print(f'  下限 (2.5% 分位数) = {ci_lower:.4f}')
print(f'  上限 (97.5% 分位数) = {ci_upper:.4f}')
print(f'  95% 自助法置信区间: ({ci_lower:.4f}, {ci_upper:.4f})')

In [ ]:
# ========== 可视化：自助法分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：自助样本均值的分布
ax1 = axes[0]
ax1.hist(bootstrap_means, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white')
ax1.axvline(x=np.mean(study_hours), color='green', linestyle='-', linewidth=2, label=f'Sample Mean = {np.mean(study_hours):.2f}')
ax1.axvline(x=ci_lower, color='red', linestyle='--', linewidth=2, label=f'95% CI Lower = {ci_lower:.2f}')
ax1.axvline(x=ci_upper, color='red', linestyle='--', linewidth=2, label=f'95% CI Upper = {ci_upper:.2f}')
ax1.set_xlabel('Bootstrap Sample Mean', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Distribution of Bootstrap Sample Means', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# 右图：原始数据 vs 自助样本
ax2 = axes[1]
ax2.scatter(range(n_micro), study_hours, s=100, color='blue', alpha=0.7, label='Original Data', zorder=5)
# 展示几个自助样本
for i in range(3):
    boot_sample = np.random.choice(study_hours, size=n_micro, replace=True)
    jitter = np.random.normal(0, 0.1, n_micro)
    ax2.scatter(range(n_micro) + jitter, boot_sample, s=50, alpha=0.3, color='red')
ax2.set_xlabel('Index', fontsize=12)
ax2.set_ylabel('Study Hours', fontsize=12)
ax2.set_title('Original Data vs Bootstrap Samples', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  左图：10000 个自助样本均值的分布，红线标示 95% 置信区间')
print(f'  右图：蓝点是原始数据，红点是自助样本（有重复和缺失）')

---

## 4. 对比传统方法与自助法

In [ ]:
# ========== 对比传统方法与自助法 ==========
# 传统方法：t 分布
n = n_micro
x_bar = np.mean(study_hours)
s = np.std(study_hours, ddof=1)
se = s / np.sqrt(n)
t_crit = stats.t.ppf(0.975, n - 1)
ci_trad_lower = x_bar - t_crit * se
ci_trad_upper = x_bar + t_crit * se

print('=' * 60)
print('📋 对比：传统方法 vs 自助法')
print('=' * 60)

print(f'\n原始数据: {study_hours}')
print(f'样本量 n = {n}')
print(f'样本均值 x̄ = {x_bar:.4f}')
print(f'样本标准差 s = {s:.4f}')

print(f'\n📊 传统方法 (t 分布):')
print(f'  t_{{crit}} = {t_crit:.4f}')
print(f'  SE = s/√n = {se:.4f}')
print(f'  95% CI: ({ci_trad_lower:.4f}, {ci_trad_upper:.4f})')
print(f'  区间宽度: {ci_trad_upper - ci_trad_lower:.4f}')

print(f'\n📊 自助法 (10000 次重抽样):')
print(f'  95% CI: ({ci_lower:.4f}, {ci_upper:.4f})')
print(f'  区间宽度: {ci_upper - ci_lower:.4f}')

print(f'\n💡 对比分析:')
print(f'  两种方法的置信区间可能略有不同，但应大致相近')
print(f'  自助法不依赖正态性假设，适用范围更广')

---

## 5. 自助法估计比例

In [ ]:
# ========== 自助法估计比例 ==========
# 场景：调查 20 人，12 人支持某政策
n_prop = 20
x_success = 12

# 创建二元数据（1=支持，0=不支持）
data_prop = np.array([1] * x_success + [0] * (n_prop - x_success))
p_hat = np.mean(data_prop)

# 自助法
n_bootstrap_prop = 10000
bootstrap_props = np.array([
    np.mean(np.random.choice(data_prop, size=n_prop, replace=True))
    for _ in range(n_bootstrap_prop)
])

# 置信区间
ci_prop_lower = np.percentile(bootstrap_props, 2.5)
ci_prop_upper = np.percentile(bootstrap_props, 97.5)

# 传统方法 (Wald)
z_95 = stats.norm.ppf(0.975)
se_prop = np.sqrt(p_hat * (1 - p_hat) / n_prop)
ci_wald_lower = p_hat - z_95 * se_prop
ci_wald_upper = p_hat + z_95 * se_prop

print('=' * 60)
print('📋 比例的置信区间：传统方法 vs 自助法')
print('=' * 60)

print(f'\n数据: {n_prop} 人中 {x_success} 人支持')
print(f'样本比例 p̂ = {p_hat:.4f}')

print(f'\n📊 传统方法 (Wald 置信区间):')
print(f'  z_{{α/2}} = {z_95:.4f}')
print(f'  SE = √(p̂q̂/n) = {se_prop:.4f}')
print(f'  95% CI: ({ci_wald_lower:.4f}, {ci_wald_upper:.4f})')

print(f'\n📊 自助法 (10000 次重抽样):')
print(f'  95% CI: ({ci_prop_lower:.4f}, {ci_prop_upper:.4f})')

# Wilson 置信区间
try:
    from statsmodels.stats.proportion import proportion_confint
    ci_wilson = proportion_confint(x_success, n_prop, alpha=0.05, method='wilson')
    print(f'\n📊 Wilson 置信区间（更准确）:')
    print(f'  95% CI: ({ci_wilson[0]:.4f}, {ci_wilson[1]:.4f})')
except ImportError:
    pass

print(f'\n💡 小样本时，Wald 区间可能不够准确，自助法和 Wilson 方法更可靠')

---

## 6. 自助法估计中位数和其他统计量

### 💡 自助法的最大优势

传统方法只能估计均值、比例、方差等特定参数。

**自助法可以估计任何统计量的置信区间！** 包括中位数、四分位数、偏度、峰度等。

In [ ]:
# ========== 自助法估计中位数 ==========
# 使用非对称数据
skewed_data = np.random.exponential(scale=5, size=30)  # 右偏数据

print('📋 右偏数据：指数分布（模拟收入数据）')
print(f'  样本量 n = {len(skewed_data)}')
print(f'  样本均值 = {np.mean(skewed_data):.2f}')
print(f'  样本中位数 = {np.median(skewed_data):.2f}')
print(f'  💡 均值 > 中位数，这是右偏分布的特征')

# 自助法估计中位数
n_boot = 10000
bootstrap_medians = np.array([
    np.median(np.random.choice(skewed_data, size=len(skewed_data), replace=True))
    for _ in range(n_boot)
])

# 自助法估计均值
bootstrap_means_skew = np.array([
    np.mean(np.random.choice(skewed_data, size=len(skewed_data), replace=True))
    for _ in range(n_boot)
])

# 置信区间
ci_median = (np.percentile(bootstrap_medians, 2.5), np.percentile(bootstrap_medians, 97.5))
ci_mean = (np.percentile(bootstrap_means_skew, 2.5), np.percentile(bootstrap_means_skew, 97.5))

print(f'\n📊 自助法 95% 置信区间:')
print(f'  中位数: ({ci_median[0]:.2f}, {ci_median[1]:.2f})')
print(f'  均值:   ({ci_mean[0]:.2f}, {ci_mean[1]:.2f})')
print(f'  \n💡 传统方法无法直接计算中位数的置信区间，自助法可以！')

In [ ]:
# ========== 可视化：中位数和均值的自助分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：中位数
ax1 = axes[0]
ax1.hist(bootstrap_medians, bins=40, density=True, alpha=0.6, color='#2ecc71', edgecolor='white')
ax1.axvline(x=np.median(skewed_data), color='blue', linestyle='-', linewidth=2, label=f'Sample Median = {np.median(skewed_data):.2f}')
ax1.axvline(x=ci_median[0], color='red', linestyle='--', linewidth=1.5)
ax1.axvline(x=ci_median[1], color='red', linestyle='--', linewidth=1.5, label='95% CI')
ax1.set_xlabel('Bootstrap Median', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Bootstrap Distribution of Median', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# 右图：均值
ax2 = axes[1]
ax2.hist(bootstrap_means_skew, bins=40, density=True, alpha=0.6, color='#3498db', edgecolor='white')
ax2.axvline(x=np.mean(skewed_data), color='blue', linestyle='-', linewidth=2, label=f'Sample Mean = {np.mean(skewed_data):.2f}')
ax2.axvline(x=ci_mean[0], color='red', linestyle='--', linewidth=1.5)
ax2.axvline(x=ci_mean[1], color='red', linestyle='--', linewidth=1.5, label='95% CI')
ax2.set_xlabel('Bootstrap Mean', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Bootstrap Distribution of Mean', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  左图：中位数的自助分布，注意分布形状可能不对称')
print(f'  右图：均值的自助分布，根据中心极限定理应近似正态')

---

## 7. 自助法估计方差和标准差

In [ ]:
# ========== 自助法估计方差 ==========
# 使用心率数据
heart_rates = np.array([76, 76, 86, 74, 66, 62, 78, 68, 62, 62, 74,
                        80, 54, 74, 74, 84, 60, 52, 84, 66, 56, 66])
n_hr = len(heart_rates)

# 自助法估计方差
n_boot_var = 10000
bootstrap_vars = np.array([
    np.var(np.random.choice(heart_rates, size=n_hr, replace=True), ddof=1)
    for _ in range(n_boot_var)
])

bootstrap_stds = np.sqrt(bootstrap_vars)

# 置信区间
ci_var = (np.percentile(bootstrap_vars, 2.5), np.percentile(bootstrap_vars, 97.5))
ci_std = (np.percentile(bootstrap_stds, 2.5), np.percentile(bootstrap_stds, 97.5))

# 传统方法（卡方）
s2 = np.var(heart_rates, ddof=1)
s = np.std(heart_rates, ddof=1)
df_hr = n_hr - 1
chi2_L = stats.chi2.ppf(0.025, df_hr)
chi2_R = stats.chi2.ppf(0.975, df_hr)
ci_chi2_var = (df_hr * s2 / chi2_R, df_hr * s2 / chi2_L)
ci_chi2_std = (np.sqrt(ci_chi2_var[0]), np.sqrt(ci_chi2_var[1]))

print('=' * 60)
print('📋 方差估计：传统方法 vs 自助法')
print('=' * 60)

print(f'\n数据: 心率数据，n = {n_hr}')
print(f'样本方差 s² = {s2:.4f}')
print(f'样本标准差 s = {s:.4f}')

print(f'\n📊 传统方法 (卡方分布):')
print(f'  方差 CI: ({ci_chi2_var[0]:.4f}, {ci_chi2_var[1]:.4f})')
print(f'  标准差 CI: ({ci_chi2_std[0]:.4f}, {ci_chi2_std[1]:.4f})')

print(f'\n📊 自助法 (10000 次重抽样):')
print(f'  方差 CI: ({ci_var[0]:.4f}, {ci_var[1]:.4f})')
print(f'  标准差 CI: ({ci_std[0]:.4f}, {ci_std[1]:.4f})')

print(f'\n💡 对比:')
print(f'  传统方法要求总体正态分布')
print(f'  自助法不需要正态性假设，但需要足够的样本量')

---

## 8. 大样本模拟：自助法的覆盖概率验证

In [ ]:
# ========== 覆盖概率模拟 ==========

mu_true = 50
sigma_true = 10
n_sample = 20
n_experiments = 1000
n_boot_each = 1000

# 存储每次实验的置信区间
ci_lows = []
ci_highs = []

for _ in range(n_experiments):
    # 从总体中抽取样本
    sample = np.random.normal(mu_true, sigma_true, n_sample)
    
    # 自助法
    boot_means = np.array([
        np.mean(np.random.choice(sample, size=n_sample, replace=True))
        for _ in range(n_boot_each)
    ])
    
    ci_lows.append(np.percentile(boot_means, 2.5))
    ci_highs.append(np.percentile(boot_means, 97.5))

ci_lows = np.array(ci_lows)
ci_highs = np.array(ci_highs)

# 计算覆盖概率
covers = (ci_lows <= mu_true) & (ci_highs >= mu_true)
coverage = covers.mean()

# 对比 t 分布方法
ci_t_lows = []
ci_t_highs = []
t_crit = stats.t.ppf(0.975, n_sample - 1)

for _ in range(n_experiments):
    sample = np.random.normal(mu_true, sigma_true, n_sample)
    x_bar = np.mean(sample)
    se = np.std(sample, ddof=1) / np.sqrt(n_sample)
    ci_t_lows.append(x_bar - t_crit * se)
    ci_t_highs.append(x_bar + t_crit * se)

ci_t_lows = np.array(ci_t_lows)
ci_t_highs = np.array(ci_t_highs)
covers_t = (ci_t_lows <= mu_true) & (ci_t_highs >= mu_true)
coverage_t = covers_t.mean()

print('=' * 60)
print('📋 覆盖概率模拟')
print('=' * 60)
print(f'\n模拟参数:')
print(f'  真实均值 μ = {mu_true}')
print(f'  样本量 n = {n_sample}')
print(f'  实验次数 = {n_experiments}')
print(f'  每次自助重抽样次数 = {n_boot_each}')

print(f'\n📊 结果:')
print(f'  自助法覆盖概率: {coverage:.4f} ({coverage*100:.1f}%)')
print(f'  t 分布覆盖概率: {coverage_t:.4f} ({coverage_t*100:.1f}%)')
print(f'  理论值: 0.9500 (95.0%)')

print(f'\n💡 两种方法的覆盖概率都应接近 95%')

In [ ]:
# ========== 可视化：前 50 个置信区间 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

n_show = 50

# 左图：自助法
ax1 = axes[0]
colors_boot = ['#2ecc71' if c else '#e74c3c' for c in covers[:n_show]]
for i in range(n_show):
    ax1.plot([ci_lows[i], ci_highs[i]], [i, i], color=colors_boot[i], linewidth=2, alpha=0.8)
ax1.axvline(x=mu_true, color='black', linestyle='--', linewidth=2, label=f'True μ = {mu_true}')
ax1.set_xlabel('Value', fontsize=12)
ax1.set_ylabel('Sample Index', fontsize=12)
ax1.set_title(f'Bootstrap 95% CI (Coverage: {covers[:n_show].sum()}/{n_show})', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图：t 分布
ax2 = axes[1]
colors_t = ['#2ecc71' if c else '#e74c3c' for c in covers_t[:n_show]]
for i in range(n_show):
    ax2.plot([ci_t_lows[i], ci_t_highs[i]], [i, i], color=colors_t[i], linewidth=2, alpha=0.8)
ax2.axvline(x=mu_true, color='black', linestyle='--', linewidth=2, label=f'True μ = {mu_true}')
ax2.set_xlabel('Value', fontsize=12)
ax2.set_ylabel('Sample Index', fontsize=12)
ax2.set_title(f't-distribution 95% CI (Coverage: {covers_t[:n_show].sum()}/{n_show})', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 图解说明：')
print(f'  绿色区间包含真实均值 μ = {mu_true}')
print(f'  红色区间不包含真实均值')
print(f'  两种方法的覆盖率都应接近 95%')

---

## 📌 核心概念回顾

### 📌 自助法 (Bootstrap)
- **定义**: 通过有放回重抽样来估计统计量变异的方法
- **原理**: 用样本数据模拟总体分布
- **含义**: 不依赖理论分布假设，适用范围广
- **步骤**: 重抽样 → 计算统计量 → 重复 → 构建 CI

### 📌 百分位数法置信区间
- **定义**: 用自助统计量的分位数构建置信区间
- **公式**: $[\text{Percentile}_{\alpha/2}, \text{Percentile}_{1-\alpha/2}]$
- **含义**: 直观简单，不需要复杂的数学推导
- **判断标准**: 重抽样次数越多，结果越稳定

### 📌 自助法的优势
- 不依赖正态性假设
- 可以估计任何统计量的置信区间
- 对小样本也适用（但效果可能不如大样本好）
- 计算简单，概念直观

### 📌 自助法的局限
- 需要计算资源（大量重抽样）
- 小样本时可能不稳定
- 不能创造不存在的信息（如果原始样本不代表总体，自助法也无法修复）

### 🔗 完整流程
```
原始样本 → 有放回重抽样 → 计算统计量 → 重复N次 → 排序 → 取分位数
    ↓            ↓             ↓           ↓        ↓         ↓
  n个数据     n个数据      均值/中位数等   1000次    排序    2.5%, 97.5%
```

### 📝 自助法 vs 传统方法

| 特性 | 传统方法 | 自助法 |
|------|---------|--------|
| 假设 | 需要分布假设 | 无分布假设 |
| 适用统计量 | 均值、比例、方差 | 任意统计量 |
| 计算 | 解析公式 | 数值模拟 |
| 小样本 | 受限 | 可用但不稳定 |
| 精确度 | 理论精确 | 近似（重抽样次数足够时很好） |

---

## ❌ 常见误区

### ❌ 误区 1: 「自助法可以弥补样本量不足」
**✓ 正确理解**: 自助法不能创造不存在的信息。如果原始样本太小或不代表总体，自助法也无法给出可靠的置信区间。自助法只是换了一种方式利用已有的样本信息。

### ❌ 误区 2: 「自助法总是比传统方法好」
**✓ 正确理解**: 当传统方法的条件满足时（如正态性成立），传统方法通常更精确。自助法的优势在于条件不满足时提供替代方案。

### ❌ 误区 3: 「重抽样次数越多越好，所以应该用无限次」
**✓ 正确理解**: 重抽样次数增加到一定程度后，精度提升很小。通常 1000-10000 次就足够了。过多的重抽样会浪费计算资源。

### ❌ 误区 4: 「自助样本的均值应该等于原始样本的均值」
**✓ 正确理解**: 由于是有放回抽样，每个自助样本的均值都可能不同。但所有自助样本均值的平均值应该接近原始样本的均值。

### ❌ 误区 5: 「自助法不需要任何假设」
**✓ 正确理解**: 自助法假设样本是简单随机样本，且样本能够代表总体。如果抽样方法有偏（如自愿样本），自助法也无法纠正这种偏差。